# 機能
- 特定のテキストを含むレコードを検索・表示する（全フィールド）
- APIでそのレコードを削除
- 保存しておいた情報を元に、新テキストで再追加(Timestamp 保持)

# pseude code
## ipynb形式で
1. tableを開く jupyterでどうやってファイルを指定するのかわからない
2. tableをpandasに変換
3. text変数を定義
4. textを含むレコードをmemory_check_vocの通り検索
5. レコードに保存されたテキストを取得(text, user_id, user_name, role, timestamp, source一応全部取得)
6. APIでレコードを削除(???どうやって)
7. 5で取得したテキストの配置で再追加

- 追加
```python
table.add([{
            "text": text,
            "user_id": user_id,
            "user_name": user_name,
            "role": role,
            "timestamp": time.time(),
            "source": "discord"
        }])
```

- 削除
```python
# text = 'テキスト'
# timestamp = 時刻
condition = text_record['text'] == text and text_record['timestamp'] == timestamp
table.delete(condition) # ここの条件の書き方がわからないな、何かしら引数が用意されているはず。
```

In [1]:
import os
print(os.getcwd())

/home/yoichi1922/src/github.com/Ichiyou1922/Mashiro_AI/brain/src/test/utility


# 検索するテキストを登録・検索

In [4]:
# 検索するテキスト
text = "ツール"

In [11]:
import lancedb
from pathlib import Path

DB_PATH = Path("../../../data")
db = lancedb.connect(str(DB_PATH))
table = db.open_table('mashiro_memory')
df = table.to_pandas()

# textを含むレコードを検索
text_records = df[df['text'].str.contains(text, na=False)]
print(f"{text}を含む記憶数: {len(text_records)}")
print(f"全記憶数: {len(df)}")
print(f"割合: {len(text_records)/len(df)*100:.1f}%")
print("=" * 50)

for i, (_, row) in enumerate(text_records.iterrows(), 1):
    print(f"# --- レコード {i} ---")
    print(f"delete_timestamp = {row['timestamp']}")
    print()
    print(f"add_data = {{")
    print(f"    'text': {repr(row['text'])},")
    print(f"    'user_id': {row['user_id']},")
    print(f"    'user_name': {repr(row['user_name'])},")
    print(f"    'role': {repr(row['role'])},")
    print(f"    'timestamp': {row['timestamp']},")
    print(f"    'source': {repr(row['source'])},")
    print(f"    'importance': {row['importance']},")
    print(f"    'last_accessed': {row['last_accessed']},")
    print(f"    'access_count': {row['access_count']},")
    print(f"    'is_reflection': {row['is_reflection']},")
    print(f"    'parent_ids': {repr(row['parent_ids'])}")
    print(f"}}")
    print()

ツールを含む記憶数: 1
全記憶数: 58
割合: 1.7%
# --- レコード 1 ---
delete_timestamp = 1770725732.8561509

add_data = {
    'text': '[ツール実行結果]\nかずはのモノマネ、最高に可愛いね！',
    'user_id': 1463583804831568024,
    'user_name': 'ましろ',
    'role': 'assistant_message',
    'timestamp': 1770725732.8561509,
    'source': 'discord',
    'importance': 0.0,
    'last_accessed': 1770725732.856152,
    'access_count': 0,
    'is_reflection': False,
    'parent_ids': ''
}



# 消去するtimestampを登録

In [12]:
delete_timestamp = 1770725732.8561509

# 消去実行

In [13]:
table.delete(f"timestamp = {delete_timestamp}")

DeleteResult(version=193)

# 確認用

In [14]:
df = table.to_pandas()
text_records = df[df['text'].str.contains(text, na=False)]
print(f"{text}を含む記憶数: {len(text_records)}")
print(f"全記憶数: {len(df)}")
print(f"割合: {len(text_records)/len(df)*100:.1f}%")
print("=" * 50)

for i, (_, row) in enumerate(text_records.iterrows(), 1):
    print(f"# --- レコード {i} ---")
    print(f"delete_timestamp = {row['timestamp']}")
    print()
    print(f"add_data = {{")
    print(f"    'text': {repr(row['text'])},")
    print(f"    'user_id': {row['user_id']},")
    print(f"    'user_name': {repr(row['user_name'])},")
    print(f"    'role': {repr(row['role'])},")
    print(f"    'timestamp': {row['timestamp']},")
    print(f"    'source': {repr(row['source'])}")
    print(f"}}")
    print()

ツールを含む記憶数: 0
全記憶数: 57
割合: 0.0%


# 再追加用データの登録・追加

In [ ]:
add_data = {
    'text': 'かずはのモノマネ、最高に可愛いね！',
    'user_id': 1463583804831568024,
    'user_name': 'ましろ',
    'role': 'assistant_message',
    'timestamp': 1770725732.8561509,
    'source': 'discord',
    'importance': 0.0,
    'last_accessed': 1770725732.856152,
    'access_count': 0,
    'is_reflection': False,
    'parent_ids': ''
}

In [10]:
table.add([add_data])

/home/yoichi1922/src/github.com/Ichiyou1922/Mashiro_AI/brain/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


AddResult(version=192)